# 06_step4_F2_applied_GCM_1deg_to_0.25deg - GCM ERA5-like 1 deg -> 0.25 deg (global)   [PLAN A]

The F2 downscaling (learned on ERA5, step3) is applied to **all GCM runs adjusted by F1** ("ERA5-like 1 deg", notebook step2), over the whole globe.

Result: `era5like_025/<run>_era5like_025.nc`. The **2025 no-hosing** one is the **target** of the RCM correction (notebook step5), where it is cropped to the RCM domain.

Volume: a global 0.25 deg daily field is ~1.5 GB per variable and per year (float32, uncompressed): runs are processed one by one.

In [ ]:
import sys, warnings
from pathlib import Path
for _p in (Path.cwd(), Path.cwd().parent):   # ba_utils.py: same folder as the notebook, or the parent folder
    sys.path.insert(0, str(_p))
import numpy as np, xarray as xr, pandas as pd
import matplotlib.pyplot as plt
import ba_utils as bu
warnings.filterwarnings("ignore")

CONFIG_PATH = None   # None = config_<profile>.json found automatically (written by notebook 00)
USE_DASK_CLUSTER = False   # True = use several CPU cores (recommended on a JEO-lab / JEO-desk node)
cfg = bu.load_config(CONFIG_PATH)
CLIENT = bu.start_dask() if USE_DASK_CLUSTER else None
VARS = cfg["variables"]                      # ['pr', 'tasmax', 'tasmin']

APPLY_TO = ["gcm_historical", "gcm_2025_nohosing", "gcm_2025_hosing", "gcm_2050_nohosing", "gcm_2050_hosing"]
OVERWRITE = False
USE_VALIDATION_F2 = False                # True: use F2 trained without 2015-2025 (tests)
BA1_DIR = bu.out_dir(cfg, "plan_A", "ba_1deg")
F2_DIR = bu.out_dir(cfg, "plan_A", "F2", "validation_without_2015_2025" if USE_VALIDATION_F2 else "full")
OUT_DIR = bu.out_dir(cfg, "plan_A", "era5like_025")
import f2_downscaler as f2

## 1. Inputs

In [ ]:
# target 0.25 deg grid = global ERA5 0.25 deg grid (the one F2 was learned on)
era5_fine = bu.read_raw(cfg, "era5", "pr").isel(time=slice(0, 1))
tlat, tlon = era5_fine.lat.values, era5_fine.lon.values
print(f"target 0.25deg grid: {len(tlat)}x{len(tlon)}")

## 2. Application of F2 (supervisor's method: `f2_downscaler.apply_f2`)

In [ ]:
written = []
for name in APPLY_TO:
    src = BA1_DIR / f"{name}_F1.nc"
    dest = OUT_DIR / f"{name}_era5like_025.nc"
    if dest.exists() and not OVERWRITE:
        print(f"[already done] {name}"); written.append(name); continue
    if not src.exists():
        print(f"[missing] {src.name}: run notebook step2"); continue
    print(f"-> {name}")
    sim_1deg = {v: bu.read_processed(src, v) for v in VARS}
    try:
        out = f2.apply_f2(sim_1deg, F2_DIR, tlat, tlon)
    except NotImplementedError as e:
        print("PLACEHOLDER:", e); break
    bu.save_ds(xr.Dataset({v: out[v] for v in VARS}), dest)
    written.append(name)

## 3. Checks

In [ ]:
for name in written:
    dest = OUT_DIR / f"{name}_era5like_025.nc"
    ds = xr.open_dataset(dest, chunks={"time": 365})
    s = ds.isel(time=slice(0, 365))
    print(f"--- {name}")
    print("grid identical to ERA5 0.25deg:", bool(bu.same_grid(ds["pr"], era5_fine)))
    print("pr min:", float(s["pr"].min()), "| frac tasmin>=tasmax (should be 0):", float(((s["tasmin"] >= s["tasmax"]) & s["tasmax"].notnull()).mean()),
          "| NaN pr:", float(s["pr"].isnull().mean()))
    # mean conservation: 1 deg (ERA5-like) vs 0.25 deg, global means over the first year
    for v in VARS:
        a = float(bu.area_mean(bu.read_processed(BA1_DIR / f"{name}_F1.nc", v).isel(time=slice(0, 365))).mean().compute())
        b = float(bu.area_mean(ds[v].isel(time=slice(0, 365))).mean().compute())
        print(f"{v:7s} global mean 1deg = {a:.3f} | 0.25deg = {b:.3f}")

name = "gcm_2025_nohosing"   # maps for the RCM target run
if name in written:
    m1 = bu.read_processed(BA1_DIR / f"{name}_F1.nc", "tasmax").isel(time=slice(0, 90)).mean("time").compute()
    m2 = bu.read_processed(OUT_DIR / f"{name}_era5like_025.nc", "tasmax").isel(time=slice(0, 90)).mean("time").compute()
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for ax, m, t in zip(axes, (m1, m2), ("tasmax 1deg (ERA5-like)", "tasmax 0.25deg (after F2)")):
        im = ax.pcolormesh(m.lon, m.lat, m, shading="auto"); plt.colorbar(im, ax=ax, shrink=0.8); ax.set_title(t)
    plt.tight_layout(); plt.show()